# 02 · Clean and split

Load May and June only. Cleaning is stateless; fixed bounds never learn from a held-out distribution.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image
from taxi_tip.config import ROOT, REPORTS, ARTIFACTS
from taxi_tip.data import read_split, clean_trips
print("Project:", ROOT)


In [ ]:
from taxi_tip.data import payment_audit
may_raw = read_split("train")
june_raw = read_split("validation")
may, may_audit = clean_trips(may_raw)
june, june_audit = clean_trips(june_raw)
display(pd.DataFrame({"month": ["May", "June"], "raw_n": [len(may_raw), len(june_raw)], "eligible_n": [len(may), len(june)], "positive_rate": [may.generous_tip.mean(), june.generous_tip.mean()]}))
display(may_audit)
display(june_audit)

## Prove the observability problem

The table below reports recorded tips. Non-card rates do not measure actual generosity. Card-only eligibility prevents cash zeros from entering the negative class.

In [ ]:
display(payment_audit(may_raw))

In [ ]:
from taxi_tip.evaluation import rate_table, SLICE_COLUMNS
for column in SLICE_COLUMNS:
    display(Markdown(f"### {column}"))
    display(rate_table(may, column))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(may.tip_pct.clip(upper=100), bins=100)
ax.axvline(25, color="orange")
ax.set(xlabel="Tip / fare (%) · >100 pooled", ylabel="Card trips", title="May tip distribution")
plt.show()

## Defend the policy

Unknown zones and nonstandard fares remain flagged. Missing passengers become unknown categories. Cash, flex fares, invalid monetary components and impossible journeys are excluded by explicit rules. Five-percentage-point rate differences are descriptive practical-effect flags; confidence intervals and sample sizes matter. July outcomes have not been examined here.